# NutriPredict Congo : Nettoyage & Fusion des données DHS
**Auteur :** MPOY Schekina Lutte-de-vie  
**Date :** 22 juin 2026  
**Notebook :** Nettoyage structurel et fusion KR + HR (02 / 05)

---

## Objectif de ce notebook

Ce notebook transforme les fichiers DHS bruts en un dataset
propre et exploitable pour l'analyse et la modélisation.

Les étapes dans l'ordre :
1. Rechargement des fichiers bruts (KR et HR)
2. Sélection des colonnes utiles
3. Traitement des valeurs aberrantes DHS (codes >= 9000)
4. Exclusion des lignes sans mesures anthropométriques
5. Fusion KR + HR sur la clé ménage
6. Construction de la variable cible (stunting)
7. Export du dataset propre

## Règle appliquée dans ce notebook
Tout ce qui peut être fait avec Pandas c-à-dire le nettoyage structurel,
sélection, recodage est fait ici. Tout ce qui doit être
"appris" sur le train set (imputation statistique, encodage,
scaling) sera fait dans le Pipeline sklearn en Phase 4 afin d'éviter toute fuite de données entre train et test.

## 1. Imports et rechargement des fichiers bruts

On recharge les fichiers  dans ce notebook.
Chaque notebook doit être autonome et ne jamais dépendre
d'un autre notebook pour avoir ses variables en mémoire.

In [1]:
import pandas as pd 
import os 
import warnings
warnings.filterwarnings("ignore")



In [2]:
# Chemins
BASE_DIR  = os.path.abspath(os.path.join(os.getcwd(), '..'))
RAW_DIR   = os.path.join(BASE_DIR, 'data', 'raw', 'congo2011_12')

CHEMIN_KR = os.path.join(RAW_DIR, 'CGKR61DT', 'CGKR61FL.DTA')
CHEMIN_HR = os.path.join(RAW_DIR, 'CGHR61DT', 'CGHR61FL.DTA')

# lecture des fichiers 
df_hr = pd.read_stata(CHEMIN_HR,convert_categoricals=False)
df_kr = pd.read_stata(CHEMIN_KR, convert_categoricals= False)

print(f" HR Chargé : {df_hr.shape[0]} lignes , {df_hr.shape[1]} colonnes")
print(f" KR Chargé : {df_kr.shape[0]} lignes , {df_kr.shape[1]} colonnes")

 HR Chargé : 11632 lignes , 3194 colonnes
 KR Chargé : 9329 lignes , 1110 colonnes


## 2. Sélection des colonnes utiles

Sur 1110 colonnes dans KR et 3194 dans HR, on ne garde
que celles qui ont un rôle direct dans le projet.

Garder toutes les colonnes poserait trois problèmes :
- Ralentir tous les traitements suivants inutilement
- Risquer d'utiliser par erreur une variable non pertinente
- Produire un dataset final illisible

In [3]:
# colonnes retenues dans kr et justification 
colonnes_kr = {
    "v001": "clés de jointures, indiquant le numéro de cluster",
    "v002": "clés de jointures, indiquant le numéro de ménage",
    "hw70": "Z-score : taille/âge x 100 (retard de croissance)",
    "hw71": "Z-score : poids/âges x 100 (insuffisance pondérale) ",
    "hw72": "Z-scores : poids/taille x 100 (émaciation)",
    "b4"  : "indique le sexe de l'enfant (1 = masculin & 2 = féminin)",
    "b8"  : "indique l'âge de l'enfant en année",
    "bord": "informe sur le rang de naissance de l'enfant"
}

# colonnes retenues dans hr et justification 

colonnes_hr = {
    "hv001" : "clés jointures, indiquant le numéro de cluster",
    "hv002" : "clés jointures, indiquant le numéro de ménage",
    "hv025" : "milieu de résidence (1=rural , 2=urbain)",
    "hv270" : "indice de richesse",
    "hv201" : "source d'eau potable",
    "hv205" : "type d'assainissement",
    "hv206" : "accès à l'électricité (0=non , 1=oui)",
    "hv219" : "sexe du chef de ménages"
}

print("=== Colonnes retenues dans KR ===")
for col, desc in colonnes_kr.items():
    print(f"  {col:<8} : {desc}")

print(f"\n=== Colonnes retenues dans HR ===")
for col, desc in colonnes_hr.items():
    print(f"  {col:<8} : {desc}")

=== Colonnes retenues dans KR ===
  v001     : clés de jointures, indiquant le numéro de cluster
  v002     : clés de jointures, indiquant le numéro de ménage
  hw70     : Z-score : taille/âge x 100 (retard de croissance)
  hw71     : Z-score : poids/âges x 100 (insuffisance pondérale) 
  hw72     : Z-scores : poids/taille x 100 (émaciation)
  b4       : indique le sexe de l'enfant (1 = masculin & 2 = féminin)
  b8       : indique l'âge de l'enfant en année
  bord     : informe sur le rang de naissance de l'enfant

=== Colonnes retenues dans HR ===
  hv001    : clés jointures, indiquant le numéro de cluster
  hv002    : clés jointures, indiquant le numéro de ménage
  hv025    : milieu de résidence (1=rural , 2=urbain)
  hv270    : indice de richesse
  hv201    : source d'eau potable
  hv205    : type d'assainissement
  hv206    : accès à l'électricité (0=non , 1=oui)
  hv219    : sexe du chef de ménages


### 2.1 Application de la sélection des colonnes

On réduit chaque dataframe aux seules colonnes retenues.
On travaille sur des copies explicites  pour ne pas
modifier les dataframes bruts originaux afin 
d'éviter les effets de bord involontaires en pandas.

In [4]:
# selection des colonnes retenues dans des copies
df_hr_sel = df_hr[list(colonnes_hr.keys())].copy()
df_kr_sel = df_kr[list(colonnes_kr.keys())].copy()

print("=== Aperçu KR ===")
print(df_kr_sel.head())

print("=== Aperçu HR ===")
print(df_hr_sel.head())

=== Aperçu KR ===
   v001  v002  hw70   hw71   hw72  b4   b8  bord
0     1     2 -67.0   68.0  167.0   2  4.0     4
1     1     4  36.0 -108.0 -200.0   1  0.0     1
2     1     5   NaN    NaN    NaN   1  1.0     4
3     1     5   NaN    NaN    NaN   2  3.0     3
4     1     6  35.0  -44.0  -78.0   2  0.0     1
=== Aperçu HR ===
   hv001  hv002  hv025  hv270  hv201  hv205  hv206  hv219
0      1      1      2      1     21     23      0      2
1      1      2      2      4     21     22      1      1
2      1      3      2      1     42     23      0      1
3      1      4      2      2     21     23      0      1
4      1      5      2      1     21     23      0      1


## 3. Traitement des valeurs aberrantes DHS sur les Z-scores

Le notebook 01 a révélé 146 codes aberrants sur hw70 :
- 9999
- 9998  
- 9996 

Ces codes ne sont pas des Z-scores réels. Si on les divise
par 100 sans les filtrer, on obtient des valeurs comme 99.99
qui fausseraient complètement le modèle ML.

Traitement : remplacer toutes les valeurs >= 9000 par NaN
sur les trois colonnes Z-score, puis diviser par 100 pour
obtenir les vrais Z-scores.

In [5]:
# traitement des codes aberrants DHS sur les trois Z-scores

for col in ["hw70","hw71","hw72"]:
    # remplacement des codes >= 9000 par NaN
    nb_aberrants = (df_kr_sel[col]>=9000).sum()
    df_kr_sel[col] = df_kr_sel[col].where(df_kr_sel[col]<9000, other=float("nan"))

    # Division par 100 pour obtenir le vrai Z-score
    df_kr_sel[col] = df_kr_sel[col]/100
    print(f"> {col} : {nb_aberrants} codes aberrants remplacés.\n  Z-scores réels entre {df_kr_sel[col].min():.2f} et {df_kr_sel[col].max():.2f} ")

> hw70 : 146 codes aberrants remplacés.
  Z-scores réels entre -5.85 et 5.71 
> hw71 : 146 codes aberrants remplacés.
  Z-scores réels entre -5.43 et 4.56 
> hw72 : 146 codes aberrants remplacés.
  Z-scores réels entre -4.94 et 4.82 


## 4. Exclusion des enfants sans mesures anthropométriques

Les enfants dont hw70 est NaN ne peuvent pas être utilisés
pour la modélisation : sans Z-score taille/âge, impossible
de déterminer s'ils présentent un retard de croissance.

On filtre uniquement sur hw70 (variable cible principale)
et non sur les trois Z-scores, pour ne pas retirer des
lignes dont hw71 ou hw72 serait manquant de manière isolée,
ce qui réduirait inutilement le dataset.

Résultat : 9 329 à 4 475 lignes (52.0% retirées).
Ce sont les 4 475 enfants avec mesure anthropométrique
valide.

In [6]:
# Exclusion des lignes sans mesure de hw70 (Z-score taille/âge)
avant = len(df_kr_sel)

df_kr_clean = df_kr_sel.dropna(subset=['hw70']).copy()

apres = len(df_kr_clean)

print(f"Lignes avant exclusion : {avant}")
print(f"Lignes après exclusion : {apres}")
print(f"Lignes retirées : {avant - apres} ({(avant-apres)/avant*100:.1f}%)")

Lignes avant exclusion : 9329
Lignes après exclusion : 4475
Lignes retirées : 4854 (52.0%)


## 5. Fusion KR + HR

On joint les caractéristiques socio-économiques du ménage
(fichier HR) à chaque enfant (fichier KR) via la clé
composite cluster_ménage construite dans le notebook 01.

Type de fusion : left join sur df_kr_clean
- on garde tous les enfants avec mesure valide
- on leur attache les données de leur ménage
- les ménages HR sans enfant mesuré disparaissent naturellement

Après fusion : le nombre de lignes doit rester 4 475.


In [7]:
# Construction des clés de jointure
df_kr_clean['cle_menage'] = (df_kr_clean['v001'].astype(str) 
                              + '_' 
                              + df_kr_clean['v002'].astype(str))

df_hr_sel['cle_menage'] = (df_hr_sel['hv001'].astype(str) 
                            + '_' 
                            + df_hr_sel['hv002'].astype(str))


# Réduction HR aux colonnes utiles + clé
# On exclut hv001/hv002 devenus redondants avec cle_menage
cols_hr_fusion = ['cle_menage', 'hv025', 'hv270', 
                  'hv201', 'hv205', 'hv206', 'hv219']

df_hr_reduit = df_hr_sel[cols_hr_fusion].drop_duplicates(subset='cle_menage')


In [8]:
# Fusion
df_fusion = df_kr_clean.merge(df_hr_reduit, on='cle_menage', how='left')

# Vérification
print(f"Lignes avant fusion : {len(df_kr_clean)}")
print(f"Lignes après fusion : {len(df_fusion)}")
print(f"Colonnes après fusion : {df_fusion.shape[1]}")
print(f"\nValeurs manquantes introduites par la fusion :")
nouvelles_manquantes = df_fusion[['hv025','hv270','hv201',
                                   'hv205','hv206','hv219']].isna().sum()
print(nouvelles_manquantes)

Lignes avant fusion : 4475
Lignes après fusion : 4475
Colonnes après fusion : 15

Valeurs manquantes introduites par la fusion :
hv025    0
hv270    0
hv201    0
hv205    0
hv206    0
hv219    0
dtype: int64


## 6. Construction de la variable cible — stunting

Le stunting (retard de croissance) est défini par l'OMS comme
un Z-score taille/âge (hw70) strictement inférieur à -2.0.

On crée une variable binaire :
- 1 = enfant en retard de croissance (hw70 < -2.0)
- 0 = enfant dans la norme (hw70 >= -2.0)

C'est cette variable que le modèle ML devra prédire.
On vérifie immédiatement la distribution pour estimer
le déséquilibre de classes, une information critique pour
le choix de la stratégie de modélisation.

In [9]:
# construction de la variable cible 
df_fusion["stunting"] = (df_fusion["hw70"]<-2.0).astype(int)

In [10]:
# Distribution
total = len(df_fusion)
n_stunting = df_fusion['stunting'].sum()
n_normal = total - n_stunting

print("=== Distribution de la variable cible ===")
print(f"Total enfants mesurés    : {total}")
print(f"Stunting (1) à risque  : {n_stunting} ({n_stunting/total*100:.1f}%)")
print(f"Normal  (0)  sans risque: {n_normal} ({n_normal/total*100:.1f}%)")

=== Distribution de la variable cible ===
Total enfants mesurés    : 4475
Stunting (1) à risque  : 1197 (26.7%)
Normal  (0)  sans risque: 3278 (73.3%)


In [11]:

print(f"\n=== Interprétation ===")
ratio = n_normal / n_stunting if n_stunting > 0 else 0
print(f"Ratio normal/stunting : {ratio:.1f} pour 1")

if n_stunting/total < 0.30:
    print(" Déséquilibre modéré détecté.")
    print("  Stratégie : class_weight='balanced' dans les modèles sklearn.")
elif n_stunting/total < 0.15:
    print(" Déséquilibre important détecté.")
    print("  Stratégie : class_weight='balanced' + évaluation SMOTE.")
else:
    print(" Distribution acceptable, pas de stratégie spéciale requise.")


=== Interprétation ===
Ratio normal/stunting : 2.7 pour 1
 Déséquilibre modéré détecté.
  Stratégie : class_weight='balanced' dans les modèles sklearn.


## 7. Export du dataset propre

Le dataset fusionné et nettoyé est exporté en CSV dans
data/processed/, le seul dossier dont le contenu peut
aller sur GitHub (car pas de données individuelles brutes DHS,
uniquement des données retraitées et anonymisées).

Ce fichier sera le point de départ de tous les notebooks
suivants.

In [12]:
# Dossier de sortie
PROCESSED_DIR = os.path.join(BASE_DIR, 'data', 'processed')
os.makedirs(PROCESSED_DIR, exist_ok=True)

# Chemin du fichier de sortie
CHEMIN_EXPORT = os.path.join(PROCESSED_DIR, 'congo_clean.csv')

# Export
df_fusion.to_csv(CHEMIN_EXPORT, index=False)

In [13]:
# Vérification
taille = os.path.getsize(CHEMIN_EXPORT) / (1024*1024)
print(f"Dataset exporté : {CHEMIN_EXPORT}")
print(f"  Taille          : {taille:.2f} Mo")
print(f"  Lignes          : {df_fusion.shape[0]}")
print(f"  Colonnes        : {df_fusion.shape[1]}")
print(f"\nColonnes présentes :")
for col in df_fusion.columns:
    print(f"  - {col}")

Dataset exporté : c:\Users\HP\Downloads\NutriPredict-Congo\data\processed\congo_clean.csv
  Taille          : 0.23 Mo
  Lignes          : 4475
  Colonnes        : 16

Colonnes présentes :
  - v001
  - v002
  - hw70
  - hw71
  - hw72
  - b4
  - b8
  - bord
  - cle_menage
  - hv025
  - hv270
  - hv201
  - hv205
  - hv206
  - hv219
  - stunting
